# NB08 — Extended Baselines, Morphometric Redundancy Ablation, Hyperparameter Audit, and Computational Cost

This notebook addresses the main model-comparison and redundancy concerns raised during manuscript review.

## Objectives
1. Add essential direct comparators: LDA, shrinkage LDA, QDA, k-NN, Random Forest, CatBoost, and TabPFN.
2. Compare **FULL_11** versus **REDUCED_7**, removing `AspectRatio`, `Solidity`, `roundness`, and `Compactness`.
3. Re-evaluate the original direct models plus LDA-SVM and LDA-XGBoost on REDUCED_7.
4. Summarize component-selection frequencies, grid-boundary selections, and computational cost.
5. Perform paired OOF ablation analyses.
6. Add cultivar-wise morphometric descriptive statistics and global group-difference tests.

FULL_11 results for the original NB03/NB04 models are reused rather than recomputed.


> ## GUÍA RÁPIDA PARA REANUDAR
> Si NB08 ya fue ejecutado y solo falta TabPFN: ve a **Sección 6**, ejecuta **CELDA 6A**, luego **CELDA 6B**, **salta Sección 7**, y ejecuta **Secciones 8–13**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, time, random, warnings, platform, re
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATA = ROOT/'01_DATA'
NOTEBOOKS = ROOT/'02_NOTEBOOKS'
RESULTS = ROOT/'03_RESULTS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'

OUT = RESULTS/'NB08_EXTENDED_ABLATION'
FIG = FIGURES/'NB08_EXTENDED_ABLATION'
TAB = TABLES/'NB08_EXTENDED_ABLATION'
LOG = LOGS/'NB08_EXTENDED_ABLATION'
for p in [OUT, FIG, TAB, LOG]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA/'INIAP_Dataset.xlsx'
BASE_METRICS = RESULTS/'NB03_BASELINES'/'baseline_metrics_by_fold.csv'
BASE_OOF = RESULTS/'NB03_BASELINES'/'baseline_oof_predictions.csv'
BASE_PARAMS = RESULTS/'NB03_BASELINES'/'baseline_best_params.csv'
HYB_METRICS = RESULTS/'NB04_HYBRIDS'/'hybrid_metrics_by_fold.csv'
HYB_OOF = RESULTS/'NB04_HYBRIDS'/'hybrid_oof_predictions.csv'
HYB_PARAMS = RESULTS/'NB04_HYBRIDS'/'hybrid_best_params.csv'

for p in [DATASET, BASE_METRICS, BASE_OOF, BASE_PARAMS, HYB_METRICS, HYB_OOF, HYB_PARAMS]:
    assert p.exists(), f'Missing required file: {p}'

SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = 'Class'
RUN_TABPFN = True
RUN_STRICT_6 = False

print('ROOT:', ROOT)
print('NB08 output:', OUT)


In [ ]:
# Required packages for extended comparators
!pip -q install xgboost catboost tabpfn statsmodels


In [ ]:
from getpass import getpass
import matplotlib.pyplot as plt
from scipy.stats import kruskal, binomtest
from statsmodels.stats.multitest import multipletests

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, log_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    TORCH_AVAILABLE = False
    DEVICE = 'cpu'

try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
except Exception as e:
    TABPFN_AVAILABLE = False
    TABPFN_IMPORT_ERROR = repr(e)

print('DEVICE:', DEVICE)
print('TabPFN import available:', TABPFN_AVAILABLE)
if RUN_TABPFN and not TABPFN_AVAILABLE:
    print('WARNING: TabPFN could not be imported. The remaining NB08 analyses will continue.')
    print(TABPFN_IMPORT_ERROR)


## 1. Dataset, feature sets, and redundancy audit

The primary ablation follows the reviewer request exactly: FULL_11 versus REDUCED_7 after removing four deterministic shape descriptors.

A separate identity check also tests whether `EquivDiameter` is numerically determined by `Area`. This is reported transparently but does not alter the pre-specified REDUCED_7 comparison.


In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
assert TARGET in df.columns
assert len(df)==4000

FEATURES_11 = [
    'Area','Perimeter','MajorAxisLength','MinorAxisLength','AspectRatio',
    'ConvexArea','EquivDiameter','Extent','Solidity','roundness','Compactness'
]
DROP_4 = ['AspectRatio','Solidity','roundness','Compactness']
FEATURES_7 = [c for c in FEATURES_11 if c not in DROP_4]
FEATURES_6 = [c for c in FEATURES_7 if c != 'EquivDiameter']

le = LabelEncoder()
y = pd.Series(le.fit_transform(df[TARGET].astype(str)), index=df.index, name=TARGET)
CLASSES = list(le.classes_)
CLASS_MAP = {int(i): str(c) for i,c in enumerate(CLASSES)}

X11 = df[FEATURES_11].copy()
X7 = df[FEATURES_7].copy()
X6 = df[FEATURES_6].copy()

identity_checks = pd.DataFrame([
    ['AspectRatio = MajorAxisLength / MinorAxisLength',
     np.max(np.abs(df.AspectRatio - df.MajorAxisLength/df.MinorAxisLength))],
    ['Solidity = Area / ConvexArea',
     np.max(np.abs(df.Solidity - df.Area/df.ConvexArea))],
    ['roundness = 4*pi*Area / Perimeter^2',
     np.max(np.abs(df['roundness'] - 4*np.pi*df.Area/(df.Perimeter**2)))],
    ['Compactness = EquivDiameter / MajorAxisLength',
     np.max(np.abs(df.Compactness - df.EquivDiameter/df.MajorAxisLength))],
    ['EquivDiameter = sqrt(4*Area/pi)',
     np.max(np.abs(df.EquivDiameter - np.sqrt(4*df.Area/np.pi)))],
], columns=['relation','max_abs_error'])
identity_checks.to_csv(TAB/'deterministic_identity_audit.csv',index=False)
display(identity_checks)

print('FULL_11:', FEATURES_11)
print('REDUCED_7:', FEATURES_7)
print('Optional STRICT_6:', FEATURES_6)
print('Class map:', CLASS_MAP)


## 2. Cultivar-wise morphometric description

Kruskal-Wallis tests provide a global four-cultivar comparison for each descriptor. Holm correction is applied across the 11 descriptors. Epsilon-squared is reported as an effect-size summary.


In [ ]:
desc_rows = []
for cls in CLASSES:
    g = df[df[TARGET].astype(str)==cls]
    for feat in FEATURES_11:
        desc_rows.append({
            'cultivar': cls,
            'feature': feat,
            'n': len(g),
            'mean': g[feat].mean(),
            'sd': g[feat].std(ddof=1),
            'median': g[feat].median(),
            'q1': g[feat].quantile(.25),
            'q3': g[feat].quantile(.75),
        })
desc = pd.DataFrame(desc_rows)
desc.to_csv(TAB/'morphometrics_by_cultivar_descriptive.csv', index=False)

kw_rows = []
for feat in FEATURES_11:
    groups = [df.loc[df[TARGET].astype(str)==cls, feat].to_numpy() for cls in CLASSES]
    H,p = kruskal(*groups)
    k=len(groups); n=sum(len(g) for g in groups)
    eps2 = max(0.0, min(1.0, (H-k+1)/(n-k)))
    kw_rows.append({'feature':feat,'kruskal_H':H,'p_raw':p,'epsilon_squared':eps2})
kw = pd.DataFrame(kw_rows)
kw['p_holm'] = multipletests(kw.p_raw, method='holm')[1]
kw['significant_holm_0.05'] = kw.p_holm < .05
kw.to_csv(TAB/'morphometrics_kruskal_holm.csv', index=False)

display(desc.head(12))
display(kw.sort_values('epsilon_squared',ascending=False))


## 3. Model definitions

Trainable models with grids are tuned only inside each outer-training fold. Fixed comparators (plain LDA, shrinkage LDA, and TabPFN) are evaluated directly on the same untouched outer folds.


In [ ]:
def model_registry(seed):
    return {
        'LogisticRegression': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',LogisticRegression(max_iter=5000, random_state=seed))
            ]),
            {'clf__C':[0.1,1,10]}
        ),
        'SVM_RBF': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',SVC(kernel='rbf',probability=True,random_state=seed))
            ]),
            {'clf__C':[1,10,100], 'clf__gamma':['scale',0.01,0.1]}
        ),
        'XGBoost': (
            XGBClassifier(
                objective='multi:softprob',eval_metric='mlogloss',
                tree_method='hist',n_jobs=1,random_state=seed
            ),
            {
                'n_estimators':[300,600],
                'max_depth':[3,5],
                'learning_rate':[0.03,0.1],
                'subsample':[0.8,1.0],
                'colsample_bytree':[0.8,1.0]
            }
        ),
        'MLP': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',MLPClassifier(max_iter=1200,early_stopping=True,random_state=seed))
            ]),
            {
                'clf__hidden_layer_sizes':[(64,),(128,64)],
                'clf__alpha':[1e-4,1e-3],
                'clf__learning_rate_init':[3e-4,1e-3]
            }
        ),
        'LDA': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',LinearDiscriminantAnalysis(solver='svd'))
            ]),
            None
        ),
        'LDA_Shrinkage': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))
            ]),
            None
        ),
        'QDA': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',QuadraticDiscriminantAnalysis())
            ]),
            {'clf__reg_param':[0.01,0.05,0.10,0.20,0.50]}
        ),
        'KNN': (
            Pipeline([
                ('scale',StandardScaler()),
                ('clf',KNeighborsClassifier())
            ]),
            {
                'clf__n_neighbors':[3,5,7,11,15],
                'clf__weights':['uniform','distance'],
                'clf__p':[1,2]
            }
        ),
        'RandomForest': (
            RandomForestClassifier(random_state=seed,n_jobs=1),
            {
                'n_estimators':[300,600],
                'max_features':['sqrt',0.7],
                'min_samples_leaf':[1,2]
            }
        ),
        'CatBoost': (
            CatBoostClassifier(
                verbose=False,random_seed=seed,loss_function='MultiClass',
                thread_count=1,allow_writing_files=False
            ),
            {
                'iterations':[300,600],
                'depth':[4,6],
                'learning_rate':[0.03,0.1]
            }
        ),
        'LDA_SVM': (
            Pipeline([
                ('scale',StandardScaler()),
                ('dr',LinearDiscriminantAnalysis()),
                ('clf',SVC(kernel='rbf',probability=True,random_state=seed))
            ]),
            {
                'dr__n_components':[1,2,3],
                'clf__C':[1,10,100],
                'clf__gamma':['scale',0.01,0.1]
            }
        ),
        'LDA_XGBoost': (
            Pipeline([
                ('scale',StandardScaler()),
                ('dr',LinearDiscriminantAnalysis()),
                ('clf',XGBClassifier(
                    objective='multi:softprob',eval_metric='mlogloss',
                    tree_method='hist',n_jobs=1,random_state=seed
                ))
            ]),
            {
                'dr__n_components':[1,2,3],
                'clf__n_estimators':[300,600],
                'clf__max_depth':[3,5],
                'clf__learning_rate':[0.03,0.1]
            }
        ),
    }

NEW_MODELS = ['LDA','LDA_Shrinkage','QDA','KNN','RandomForest','CatBoost']
REDUCED_RERUN_MODELS = [
    'LogisticRegression','SVM_RBF','XGBoost','MLP','LDA_SVM','LDA_XGBoost'
]
print('New full/reduced models:', NEW_MODELS)
print('Existing models rerun on REDUCED_7:', REDUCED_RERUN_MODELS)


In [ ]:
def metric_dict(y_true,y_pred,y_prob=None):
    out={
        'f1_macro':f1_score(y_true,y_pred,average='macro'),
        'accuracy':accuracy_score(y_true,y_pred),
        'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
        'precision_macro':precision_score(y_true,y_pred,average='macro',zero_division=0),
        'recall_macro':recall_score(y_true,y_pred,average='macro',zero_division=0),
        'mcc':matthews_corrcoef(y_true,y_pred),
        'kappa':cohen_kappa_score(y_true,y_pred)
    }
    if y_prob is not None:
        try:
            out['log_loss']=log_loss(y_true,y_prob,labels=np.arange(len(CLASSES)))
        except Exception:
            out['log_loss']=np.nan
    else:
        out['log_loss']=np.nan
    return out

def safe_name(s):
    return re.sub(r'[^A-Za-z0-9_\-]+','_',s)

def load_checkpoint(prefix):
    fm=OUT/f'{prefix}_metrics_CHECKPOINT.csv'
    fp=OUT/f'{prefix}_oof_CHECKPOINT.csv'
    fb=OUT/f'{prefix}_params_CHECKPOINT.csv'
    metrics=pd.read_csv(fm) if fm.exists() else pd.DataFrame()
    preds=pd.read_csv(fp) if fp.exists() else pd.DataFrame()
    params=pd.read_csv(fb) if fb.exists() else pd.DataFrame()
    return metrics,preds,params

def save_checkpoint(prefix,metrics,preds,params):
    metrics.to_csv(OUT/f'{prefix}_metrics_CHECKPOINT.csv',index=False)
    preds.to_csv(OUT/f'{prefix}_oof_CHECKPOINT.csv',index=False)
    params.to_csv(OUT/f'{prefix}_params_CHECKPOINT.csv',index=False)

def completed_keys(metrics):
    if metrics.empty:
        return set()
    return set(zip(metrics['seed'].astype(int),metrics['outer_fold'].astype(int)))

def evaluate_nested(model_name,feature_set,X,y):
    prefix=safe_name(f'{feature_set}_{model_name}')
    metrics,preds,params=load_checkpoint(prefix)
    done=completed_keys(metrics)

    for seed in SEEDS:
        outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=seed)
        for fold,(tr,te) in enumerate(outer.split(X,y),start=1):
            if (seed,fold) in done:
                print(f'SKIP completed: {feature_set} | {model_name} | seed {seed} fold {fold}')
                continue

            print(f'RUN {feature_set} | {model_name} | seed {seed} | fold {fold}/{OUTER_FOLDS}',flush=True)
            est,grid=model_registry(seed+fold)[model_name]
            Xtr=X.iloc[tr]; Xte=X.iloc[te]
            ytr=y.iloc[tr]; yte=y.iloc[te]

            t0=time.time()
            if grid is None:
                best=clone(est)
                best.fit(Xtr,ytr)
                fit_search_seconds=time.time()-t0
                refit_seconds=fit_search_seconds
                best_score_inner=np.nan
                best_params={}
            else:
                inner=StratifiedKFold(
                    n_splits=INNER_FOLDS,shuffle=True,random_state=seed+fold
                )
                search=GridSearchCV(
                    est,grid,scoring='f1_macro',cv=inner,n_jobs=-1,
                    refit=True,return_train_score=False,error_score='raise'
                )
                search.fit(Xtr,ytr)
                fit_search_seconds=time.time()-t0
                refit_seconds=float(getattr(search,'refit_time_',np.nan))
                best=search.best_estimator_
                best_score_inner=float(search.best_score_)
                best_params=search.best_params_

            tp=time.time()
            yp=best.predict(Xte)
            predict_seconds=time.time()-tp

            yprob=None
            proba_seconds=np.nan
            if hasattr(best,'predict_proba'):
                tp=time.time()
                yprob=best.predict_proba(Xte)
                proba_seconds=time.time()-tp

            met=metric_dict(yte,yp,yprob)
            met.update({
                'feature_set':feature_set,'model':model_name,
                'seed':seed,'outer_fold':fold,'n_train':len(tr),'n_test':len(te),
                'fit_search_seconds':fit_search_seconds,
                'refit_seconds':refit_seconds,
                'predict_seconds':predict_seconds,
                'predict_proba_seconds':proba_seconds
            })
            metrics=pd.concat([metrics,pd.DataFrame([met])],ignore_index=True)

            par={
                'feature_set':feature_set,'model':model_name,
                'seed':seed,'outer_fold':fold,
                'best_score_inner':best_score_inner,
                'best_params':json.dumps(best_params)
            }
            params=pd.concat([params,pd.DataFrame([par])],ignore_index=True)

            rows=[]
            yte_arr=yte.to_numpy()
            for j,rid in enumerate(te):
                rec={
                    'row_id':int(rid),
                    'y_true':CLASSES[int(yte_arr[j])],
                    'y_pred':CLASSES[int(yp[j])],
                    'feature_set':feature_set,'model':model_name,
                    'seed':seed,'outer_fold':fold
                }
                if yprob is not None:
                    classes_model=getattr(best,'classes_',np.arange(len(CLASSES)))
                    for kk,cc in enumerate(classes_model):
                        rec[f'prob_{CLASSES[int(cc)]}']=float(yprob[j,kk])
                rows.append(rec)
            preds=pd.concat([preds,pd.DataFrame(rows)],ignore_index=True)

            save_checkpoint(prefix,metrics,preds,params)

    assert len(metrics)==len(SEEDS)*OUTER_FOLDS, (feature_set,model_name,len(metrics))
    assert preds.groupby(['seed']).row_id.nunique().eq(4000).all(), (feature_set,model_name)
    metrics.to_csv(OUT/f'{prefix}_metrics.csv',index=False)
    preds.to_csv(OUT/f'{prefix}_oof.csv',index=False)
    params.to_csv(OUT/f'{prefix}_params.csv',index=False)
    return metrics,preds,params


## 4. New classical comparators on FULL_11 and REDUCED_7


In [ ]:
all_metrics=[]
all_preds=[]
all_params=[]

for feature_set,X in [('FULL_11',X11),('REDUCED_7',X7)]:
    for model in NEW_MODELS:
        m,p,b=evaluate_nested(model,feature_set,X,y)
        all_metrics.append(m); all_preds.append(p); all_params.append(b)

print('Completed new classical comparators.')


## 5. Original direct models and strongest LDA hybrids on REDUCED_7

FULL_11 results already exist in NB03/NB04. Only REDUCED_7 is recomputed here.


In [ ]:
for model in REDUCED_RERUN_MODELS:
    m,p,b=evaluate_nested(model,'REDUCED_7',X7,y)
    all_metrics.append(m); all_preds.append(p); all_params.append(b)

print('Completed REDUCED_7 reruns.')


## 6. TabPFN fixed pretrained comparator

> 🟢 **EMPEZAR AQUÍ PARA TERMINAR TABPFN**
>
> Si ya ejecutaste las Secciones 1–5 anteriormente, **NO vuelvas a entrenar los otros modelos**.
> Para cerrar NB08 debes ejecutar solamente las **dos celdas siguientes**:
>
> 1. **CELDA 6A — AUTENTICACIÓN:** pega tu API Key de Prior Labs cuando Colab la solicite.
> 2. **CELDA 6B — TABPFN:** ejecuta la función completa para FULL_11 y REDUCED_7.
>
> Requisito previo: haber aceptado la licencia correspondiente en Prior Labs. La API Key **no se guarda en Drive ni queda escrita en el notebook**.


In [ ]:
# ============================================================
# 🟢 CELDA 6A — EJECUTAR ESTA CELDA PRIMERO
# Pega tu TABPFN_TOKEN únicamente cuando aparezca el cuadro oculto.
# NO escribas la clave directamente en este código.
# ============================================================
# Secure TabPFN authentication — token is kept in memory only.
if RUN_TABPFN and TABPFN_AVAILABLE:
    token = os.environ.get('TABPFN_TOKEN', '').strip()
    if not token:
        token = getpass('Paste TABPFN_TOKEN (input hidden; not saved): ').strip()
        if not token:
            raise RuntimeError(
                'TABPFN_TOKEN was not provided. Accept the TabPFN license and paste your API key.'
            )
        os.environ['TABPFN_TOKEN'] = token
    print('TabPFN authentication token loaded in memory only.')
else:
    print('TabPFN authentication not required because TabPFN is disabled or unavailable.')


In [ ]:
# ============================================================
# 🟢 CELDA 6B — EJECUTAR ESTA CELDA DESPUÉS DE 6A
# Ejecuta TabPFN para FULL_11 y REDUCED_7.
# Esta celda usa checkpoints: si algo ya está terminado, lo omite.
# ============================================================
def evaluate_tabpfn(feature_set,X,y):
    if not RUN_TABPFN:
        print('TabPFN disabled.')
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    if not TABPFN_AVAILABLE:
        print('TabPFN unavailable; skipping without aborting NB08.')
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()

    prefix=safe_name(f'{feature_set}_TabPFN')
    metrics,preds,params=load_checkpoint(prefix)
    done=completed_keys(metrics)

    for seed in SEEDS:
        outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=seed)
        for fold,(tr,te) in enumerate(outer.split(X,y),start=1):
            if (seed,fold) in done:
                print(f'SKIP completed: {feature_set} | TabPFN | seed {seed} fold {fold}')
                continue

            print(f'RUN {feature_set} | TabPFN | seed {seed} | fold {fold}/{OUTER_FOLDS}',flush=True)
            Xtr=X.iloc[tr].to_numpy(np.float32)
            Xte=X.iloc[te].to_numpy(np.float32)
            ytr=y.iloc[tr].to_numpy()
            yte=y.iloc[te].to_numpy()

            try:
                clf=TabPFNClassifier(device=DEVICE)
            except TypeError:
                clf=TabPFNClassifier()

            t0=time.time()
            clf.fit(Xtr,ytr)
            fit_seconds=time.time()-t0

            tp=time.time()
            yp=clf.predict(Xte)
            predict_seconds=time.time()-tp

            tp=time.time()
            yprob=clf.predict_proba(Xte)
            proba_seconds=time.time()-tp

            met=metric_dict(yte,yp,yprob)
            met.update({
                'feature_set':feature_set,'model':'TabPFN',
                'seed':seed,'outer_fold':fold,'n_train':len(tr),'n_test':len(te),
                'fit_search_seconds':fit_seconds,'refit_seconds':fit_seconds,
                'predict_seconds':predict_seconds,'predict_proba_seconds':proba_seconds
            })
            metrics=pd.concat([metrics,pd.DataFrame([met])],ignore_index=True)
            params=pd.concat([params,pd.DataFrame([{
                'feature_set':feature_set,'model':'TabPFN','seed':seed,'outer_fold':fold,
                'best_score_inner':np.nan,'best_params':'{}'
            }])],ignore_index=True)

            rows=[]
            classes_model=getattr(clf,'classes_',np.arange(len(CLASSES)))
            for j,rid in enumerate(te):
                rec={
                    'row_id':int(rid),
                    'y_true':CLASSES[int(yte[j])],
                    'y_pred':CLASSES[int(yp[j])],
                    'feature_set':feature_set,'model':'TabPFN',
                    'seed':seed,'outer_fold':fold
                }
                for kk,cc in enumerate(classes_model):
                    rec[f'prob_{CLASSES[int(cc)]}']=float(yprob[j,kk])
                rows.append(rec)
            preds=pd.concat([preds,pd.DataFrame(rows)],ignore_index=True)
            save_checkpoint(prefix,metrics,preds,params)

    if not metrics.empty:
        assert len(metrics)==15
        assert preds.groupby('seed').row_id.nunique().eq(4000).all()
        metrics.to_csv(OUT/f'{prefix}_metrics.csv',index=False)
        preds.to_csv(OUT/f'{prefix}_oof.csv',index=False)
        params.to_csv(OUT/f'{prefix}_params.csv',index=False)
        stale_failure = LOG/f'tabpfn_failure_{feature_set}.txt'
        if stale_failure.exists():
            stale_failure.unlink()
    return metrics,preds,params

for feature_set,X in [('FULL_11',X11),('REDUCED_7',X7)]:
    try:
        m,p,b=evaluate_tabpfn(feature_set,X,y)
        if not m.empty:
            all_metrics.append(m); all_preds.append(p); all_params.append(b)
    except Exception as e:
        print(f'WARNING: TabPFN failed for {feature_set}: {repr(e)}')
        with open(LOG/f'tabpfn_failure_{feature_set}.txt','w') as f:
            f.write(repr(e))

print('TabPFN section completed.')


## 7. Optional STRICT_6 sensitivity analysis

> 🔴 **NO EJECUTAR ESTA SECCIÓN AHORA.**
>
> Mantener `RUN_STRICT_6 = False`. Este análisis es opcional y no forma parte del cierre principal de NB08.
> Después de terminar TabPFN en la Sección 6, **salta directamente a la Sección 8**.

`EquivDiameter` is additionally checked as an exact area-derived variable. STRICT_6 is optional and OFF by default to keep the primary analysis aligned with the four-variable removal.


In [ ]:
if RUN_STRICT_6:
    for model in ['LDA','SVM_RBF','XGBoost','LDA_XGBoost']:
        m,p,b=evaluate_nested(model,'STRICT_6',X6,y)
        all_metrics.append(m); all_preds.append(p); all_params.append(b)
else:
    print('STRICT_6 not executed.')


## 8. Consolidate NB08 results

> 🟢 **DESPUÉS DE TERMINAR TABPFN, EJECUTAR DESDE AQUÍ HASTA EL FINAL DEL NOTEBOOK.**
>
> Esta parte reconstruye las tablas y resúmenes usando los resultados ya guardados en Drive. No vuelve a entrenar los modelos principales.


In [ ]:
# Restart-safe consolidation.
# Reconstruct NB08 from completed files on Drive instead of depending on in-memory lists.
metric_files = sorted([
    p for p in OUT.glob('*_metrics.csv')
    if '_CHECKPOINT' not in p.name and p.name != 'nb08_metrics_by_fold.csv'
])
oof_files = sorted([
    p for p in OUT.glob('*_oof.csv')
    if '_CHECKPOINT' not in p.name and p.name != 'nb08_oof_predictions.csv'
])
param_files = sorted([
    p for p in OUT.glob('*_params.csv')
    if '_CHECKPOINT' not in p.name and p.name != 'nb08_best_params.csv'
])

if not metric_files:
    raise RuntimeError('No completed NB08 metric files were found on Drive.')

metrics_nb08 = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
preds_nb08 = pd.concat([pd.read_csv(p) for p in oof_files], ignore_index=True) if oof_files else pd.DataFrame()
params_nb08 = pd.concat([pd.read_csv(p) for p in param_files], ignore_index=True) if param_files else pd.DataFrame()

# Remove accidental duplicates if a section was rerun.
metrics_nb08 = metrics_nb08.drop_duplicates(['feature_set','model','seed','outer_fold']).reset_index(drop=True)
if not preds_nb08.empty:
    preds_nb08 = preds_nb08.drop_duplicates(
        ['feature_set','model','seed','outer_fold','row_id']
    ).reset_index(drop=True)
if not params_nb08.empty:
    params_nb08 = params_nb08.drop_duplicates(
        ['feature_set','model','seed','outer_fold']
    ).reset_index(drop=True)

metrics_nb08.to_csv(OUT/'nb08_metrics_by_fold.csv',index=False)
preds_nb08.to_csv(OUT/'nb08_oof_predictions.csv',index=False)
params_nb08.to_csv(OUT/'nb08_best_params.csv',index=False)

summary_nb08=metrics_nb08.groupby(['feature_set','model']).agg(
    mean_macro_f1=('f1_macro','mean'),
    sd_macro_f1=('f1_macro','std'),
    mean_accuracy=('accuracy','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_kappa=('kappa','mean'),
    mean_log_loss=('log_loss','mean'),
    mean_refit_seconds=('refit_seconds','mean'),
    mean_predict_seconds=('predict_seconds','mean'),
    mean_predict_proba_seconds=('predict_proba_seconds','mean')
).reset_index()
summary_nb08.to_csv(TAB/'nb08_model_summary.csv',index=False)

print(f'Loaded {len(metric_files)} completed model/feature-set result files from Drive.')
print('TabPFN completed combinations:',
      summary_nb08.loc[summary_nb08.model.eq('TabPFN'), ['feature_set','model']].to_dict('records'))
display(summary_nb08.sort_values(['feature_set','mean_macro_f1'],ascending=[True,False]))


## 9. Expanded FULL_11 comparison


In [ ]:
old_base=pd.read_csv(BASE_METRICS)
old_hyb=pd.read_csv(HYB_METRICS)
deep_path=RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_metrics_by_fold.csv'
old_deep=pd.read_csv(deep_path) if deep_path.exists() else pd.DataFrame()

old_full=pd.concat([old_base,old_hyb,old_deep],ignore_index=True)
old_full['feature_set']='FULL_11'

new_full=metrics_nb08[metrics_nb08.feature_set=='FULL_11'].copy()
expanded_full=pd.concat([old_full,new_full],ignore_index=True,sort=False)

expanded_summary=expanded_full.groupby('model').agg(
    mean_macro_f1=('f1_macro','mean'),
    sd_macro_f1=('f1_macro','std'),
    mean_accuracy=('accuracy','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_kappa=('kappa','mean'),
    mean_log_loss=('log_loss','mean')
).sort_values('mean_macro_f1',ascending=False)

expanded_summary.to_csv(TAB/'expanded_FULL11_model_summary.csv')
display(expanded_summary)

plot=expanded_summary.reset_index()
plt.figure(figsize=(11,6))
plt.errorbar(np.arange(len(plot)),plot.mean_macro_f1,yerr=plot.sd_macro_f1,fmt='o',capsize=4)
plt.xticks(np.arange(len(plot)),plot.model,rotation=55,ha='right')
plt.ylabel('Macro-F1')
plt.tight_layout()
plt.savefig(FIG/'expanded_FULL11_macro_f1.png',dpi=300,bbox_inches='tight')
plt.show()


## 10. Hyperparameter and component-selection audit


In [ ]:
base_params=pd.read_csv(BASE_PARAMS)
hyb_params=pd.read_csv(HYB_PARAMS)

def parse_jsonish(x):
    if isinstance(x,dict):
        return x
    try:
        return json.loads(x)
    except Exception:
        return {}

def explode_params(df_params,source):
    rows=[]
    for _,r in df_params.iterrows():
        d=parse_jsonish(r.get('best_params','{}'))
        for k,v in d.items():
            rows.append({
                'source':source,
                'feature_set':r.get('feature_set','FULL_11'),
                'model':r['model'],'seed':int(r['seed']),'outer_fold':int(r['outer_fold']),
                'parameter':k,'value':str(v)
            })
    return pd.DataFrame(rows)

parts=[explode_params(base_params,'NB03'),explode_params(hyb_params,'NB04')]
if not params_nb08.empty:
    parts.append(explode_params(params_nb08,'NB08'))
long_params=pd.concat(parts,ignore_index=True)
long_params.to_csv(TAB/'selected_hyperparameters_long.csv',index=False)

freq=(long_params.groupby(['source','feature_set','model','parameter','value'])
      .size().reset_index(name='count'))
freq.to_csv(TAB/'selected_hyperparameter_frequency.csv',index=False)

comp=freq[freq.parameter=='dr__n_components'].copy()
comp.to_csv(TAB/'component_selection_frequency.csv',index=False)

svm_c=freq[freq.parameter=='clf__C'].copy()
svm_c.to_csv(TAB/'svm_C_selection_frequency.csv',index=False)

boundary_rows=[]
for source,feature_set,model,param,maxval in [
    ('NB03','FULL_11','LogisticRegression','clf__C','10'),
    ('NB03','FULL_11','SVM_RBF','clf__C','100'),
    ('NB04','FULL_11','PCA_SVM','dr__n_components','10'),
    ('NB04','FULL_11','PCA_MLP','dr__n_components','10'),
    ('NB04','FULL_11','PCA_XGBoost','dr__n_components','10'),
    ('NB04','FULL_11','LDA_SVM','dr__n_components','3'),
    ('NB04','FULL_11','LDA_MLP','dr__n_components','3'),
    ('NB04','FULL_11','LDA_XGBoost','dr__n_components','3'),
]:
    z=long_params[(long_params.source==source)&(long_params.feature_set==feature_set)&
                  (long_params.model==model)&(long_params.parameter==param)]
    boundary_rows.append({
        'source':source,'feature_set':feature_set,'model':model,'parameter':param,
        'boundary_value':maxval,'selected_at_boundary_n':int((z.value==maxval).sum()),
        'total_outer_folds':int(len(z)),
        'fraction_at_boundary':float((z.value==maxval).mean()) if len(z) else np.nan,
        'note':'For LDA, 3 components is the theoretical C-1 maximum for four classes.'
               if param=='dr__n_components' and maxval=='3' else ''
    })
boundary=pd.DataFrame(boundary_rows)
boundary.to_csv(TAB/'grid_boundary_audit.csv',index=False)

display(comp)
display(boundary)


## 11. Paired FULL_11 vs REDUCED_7 ablation

For every model available under both feature sets, OOF predictions are aligned by grain and seed.

**Δ macro-F1 = REDUCED_7 − FULL_11**

Paired bootstrap: 3,000 resamples per seed. McNemar p-values are Holm-adjusted across all model-seed ablation tests.


In [ ]:
old_base_oof=pd.read_csv(BASE_OOF); old_base_oof['feature_set']='FULL_11'
old_hyb_oof=pd.read_csv(HYB_OOF); old_hyb_oof['feature_set']='FULL_11'
deep_oof_path=RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_oof_predictions.csv'
old_deep_oof=pd.read_csv(deep_oof_path) if deep_oof_path.exists() else pd.DataFrame()
if not old_deep_oof.empty:
    old_deep_oof['feature_set']='FULL_11'

all_oof=pd.concat([old_base_oof,old_hyb_oof,old_deep_oof,preds_nb08],ignore_index=True,sort=False)

models_both=[]
for model in sorted(all_oof.model.unique()):
    fs=set(all_oof.loc[all_oof.model==model,'feature_set'].dropna().unique())
    if {'FULL_11','REDUCED_7'}.issubset(fs):
        models_both.append(model)
print('Models with FULL_11 and REDUCED_7:',models_both)

B=3000
abl_rows=[]

for model in models_both:
    for seed in SEEDS:
        a=(all_oof[(all_oof.model==model)&(all_oof.feature_set=='FULL_11')&
                   (all_oof.seed==seed)]
           .drop_duplicates('row_id').sort_values('row_id'))
        b=(all_oof[(all_oof.model==model)&(all_oof.feature_set=='REDUCED_7')&
                   (all_oof.seed==seed)]
           .drop_duplicates('row_id').sort_values('row_id'))
        if len(a)!=4000 or len(b)!=4000:
            continue

        assert np.array_equal(a.row_id.to_numpy(),b.row_id.to_numpy())
        assert np.array_equal(a.y_true.astype(str).to_numpy(),b.y_true.astype(str).to_numpy())

        yt=a.y_true.astype(str).to_numpy()
        pa=a.y_pred.astype(str).to_numpy()
        pb=b.y_pred.astype(str).to_numpy()

        f_full=f1_score(yt,pa,average='macro')
        f_red=f1_score(yt,pb,average='macro')
        delta=f_red-f_full

        rng=np.random.default_rng(seed + sum(ord(c) for c in model))
        ds=np.empty(B)
        n=len(yt)
        for i in range(B):
            ix=rng.integers(0,n,n)
            ds[i]=f1_score(yt[ix],pb[ix],average='macro')-f1_score(yt[ix],pa[ix],average='macro')
        lo,hi=np.quantile(ds,[.025,.975])

        ca=(pa==yt); cb=(pb==yt)
        n01=int(np.sum(ca & ~cb))
        n10=int(np.sum(~ca & cb))
        disc=n01+n10
        p_mc=1.0 if disc==0 else binomtest(min(n01,n10),disc,p=.5,alternative='two-sided').pvalue

        abl_rows.append({
            'model':model,'seed':seed,
            'macro_f1_full11':f_full,'macro_f1_reduced7':f_red,
            'delta_reduced_minus_full':delta,
            'bootstrap_ci_low':lo,'bootstrap_ci_high':hi,
            'ci_excludes_zero':bool(lo>0 or hi<0),
            'ci_within_pm_0.01':bool(lo>=-0.01 and hi<=0.01),
            'mcnemar_full_correct_reduced_wrong':n01,
            'mcnemar_full_wrong_reduced_correct':n10,
            'mcnemar_p_raw':p_mc
        })

abl=pd.DataFrame(abl_rows)
if len(abl):
    abl['mcnemar_p_holm']=multipletests(abl.mcnemar_p_raw,method='holm')[1]
    abl['mcnemar_holm_sig_0.05']=abl.mcnemar_p_holm<.05
abl.to_csv(TAB/'feature_ablation_paired_by_seed.csv',index=False)

abl_summary=abl.groupby('model').agg(
    mean_full11=('macro_f1_full11','mean'),
    mean_reduced7=('macro_f1_reduced7','mean'),
    mean_delta=('delta_reduced_minus_full','mean'),
    min_ci_low=('bootstrap_ci_low','min'),
    max_ci_high=('bootstrap_ci_high','max'),
    seeds_ci_excludes_zero=('ci_excludes_zero','sum'),
    seeds_ci_within_pm_0_01=('ci_within_pm_0.01','sum'),
    seeds_mcnemar_holm_sig=('mcnemar_holm_sig_0.05','sum')
).sort_values('mean_delta',ascending=False)
abl_summary.to_csv(TAB/'feature_ablation_summary.csv')
display(abl_summary)


In [ ]:
if len(abl_summary):
    p=abl_summary.reset_index()
    plt.figure(figsize=(10,5.8))
    for _,row in p.iterrows():
        plt.plot([0,1],[row.mean_full11,row.mean_reduced7],marker='o',alpha=.8)
        plt.text(1.02,row.mean_reduced7,row['model'],fontsize=8,va='center')
    plt.xticks([0,1],['FULL_11','REDUCED_7'])
    plt.ylabel('Mean OOF Macro-F1 across seeds')
    plt.xlim(-.15,1.55)
    plt.tight_layout()
    plt.savefig(FIG/'feature_ablation_11_vs_7_macro_f1.png',dpi=300,bbox_inches='tight')
    plt.show()


## 12. Computational-cost table


In [ ]:
runtime=metrics_nb08.groupby(['feature_set','model']).agg(
    mean_search_fit_seconds=('fit_search_seconds','mean'),
    sd_search_fit_seconds=('fit_search_seconds','std'),
    mean_final_refit_seconds=('refit_seconds','mean'),
    mean_predict_seconds_800=('predict_seconds','mean'),
    mean_predict_proba_seconds_800=('predict_proba_seconds','mean')
).reset_index()

runtime['mean_predict_ms_per_grain'] = runtime['mean_predict_seconds_800']/800*1000
runtime.to_csv(TAB/'computational_cost_nb08.csv',index=False)
display(runtime.sort_values(['feature_set','mean_predict_ms_per_grain']))

z=runtime[runtime.feature_set=='FULL_11'].sort_values('mean_final_refit_seconds')
plt.figure(figsize=(10,6))
plt.barh(z.model,z.mean_final_refit_seconds)
plt.xlabel('Mean final refit time per outer fold (s)')
plt.tight_layout()
plt.savefig(FIG/'full11_refit_time_new_comparators.png',dpi=300,bbox_inches='tight')
plt.show()


## 13. Integrity checks and run log


In [ ]:
expected_new = {(fs,m) for fs in ['FULL_11','REDUCED_7'] for m in NEW_MODELS}
expected_reduced = {('REDUCED_7',m) for m in REDUCED_RERUN_MODELS}
observed = set(zip(metrics_nb08.feature_set,metrics_nb08.model))

missing_core = sorted((expected_new | expected_reduced) - observed)
assert not missing_core, f'Missing core NB08 model/feature-set combinations: {missing_core}'

for (fs,m),g in metrics_nb08.groupby(['feature_set','model']):
    assert len(g)==15, (fs,m,len(g))

tabpfn_completed = (
    ('FULL_11','TabPFN') in observed and ('REDUCED_7','TabPFN') in observed
)

run_info={
    'seeds':SEEDS,
    'outer_folds':OUTER_FOLDS,
    'inner_folds':INNER_FOLDS,
    'full_features':FEATURES_11,
    'reduced7_features':FEATURES_7,
    'removed_for_reviewer_ablation':DROP_4,
    'strict6_features_optional':FEATURES_6,
    'run_strict6':RUN_STRICT_6,
    'new_models_full_and_reduced':NEW_MODELS,
    'existing_models_rerun_reduced7':REDUCED_RERUN_MODELS,
    'tabpfn_requested':RUN_TABPFN,
    'tabpfn_import_available':TABPFN_AVAILABLE,
    'tabpfn_completed_full_and_reduced':tabpfn_completed,
    'bootstrap_resamples_ablation_per_seed':3000,
    'ablation_delta_definition':'REDUCED_7 minus FULL_11 macro-F1',
    'mcnemar_correction':'Holm across all model-seed ablation tests',
    'device':DEVICE,
    'note':(
        'FULL_11 results for original NB03/NB04 models were reused rather than recomputed. '
        'REDUCED_7 reruns were independently tuned inside each outer-training fold.'
    )
}
with open(LOG/'NB08_run_info.json','w') as f:
    json.dump(run_info,f,indent=2)

import sklearn, scipy, xgboost, catboost
versions={
    'python':sys.version,
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'scikit_learn':sklearn.__version__,
    'scipy':scipy.__version__,
    'xgboost':xgboost.__version__,
    'catboost':catboost.__version__,
}
try:
    import tabpfn
    versions['tabpfn']=getattr(tabpfn,'__version__','unknown')
except Exception:
    versions['tabpfn']='unavailable'
if TORCH_AVAILABLE:
    versions['torch']=torch.__version__
with open(LOG/'NB08_package_versions.json','w') as f:
    json.dump(versions,f,indent=2)

print('Core NB08 integrity checks PASSED.')
print('TabPFN completed:', tabpfn_completed)
print('NB08 completed successfully.')


### Main expected outputs

**Results**
- `03_RESULTS/NB08_EXTENDED_ABLATION/nb08_metrics_by_fold.csv`
- `03_RESULTS/NB08_EXTENDED_ABLATION/nb08_oof_predictions.csv`
- `03_RESULTS/NB08_EXTENDED_ABLATION/nb08_best_params.csv`

**Tables**
- `morphometrics_by_cultivar_descriptive.csv`
- `morphometrics_kruskal_holm.csv`
- `deterministic_identity_audit.csv`
- `expanded_FULL11_model_summary.csv`
- `feature_ablation_paired_by_seed.csv`
- `feature_ablation_summary.csv`
- `component_selection_frequency.csv`
- `grid_boundary_audit.csv`
- `computational_cost_nb08.csv`

**Figures**
- `expanded_FULL11_macro_f1.png`
- `feature_ablation_11_vs_7_macro_f1.png`
- `full11_refit_time_new_comparators.png`

NB09 can then focus on corrected-resampling inference, formal/sensitivity equivalence testing, reliability diagrams, and accuracy–coverage selective classification.


If TabPFN was completed after the first NB08 run, rerun Sections 8–13 to refresh all consolidated summaries and logs.
